# Track 11 — Google Open Knowledge Format (OKF)
## Authoritative Schemas, Progressive Disclosure, Knowledge Graphs, and Provenance-Grounded RAG

---

### Executive Overview & Theoretical Motivation

In modern enterprise AI systems, autonomous agents and LLMs face the **context-assembly crisis**:
1. **Unstructured Vector Blobs & Boundary Loss:** Naive semantic chunking splits data into arbitrary window sizes (e.g., 512 tokens), severing data dictionaries, schema constraints, and business formulas.
2. **Loss of Provenance & Attestation:** Once text is embedded into a dense vector space, critical metadata—such as *who* authored the rule, *when* it was verified, and *what* physical system it maps to—is discarded or buried in lossy embeddings.
3. **Context Window Saturation:** Feeding entire data warehouses or monolithic wikis into agent prompts instantly exhausts context token limits and induces "lost-in-the-middle" attention degradation.

To resolve these challenges, Google Cloud introduced the **Open Knowledge Format (OKF)** (originating from tech leads Sam McVeety and Amir Hormati). OKF formalizes the **LLM-Wiki pattern** into an open, vendor-neutral, file-based specification:
- **Plain Markdown Files (`.md`):** Human-readable in any text editor, versionable in standard Git repositories, zero proprietary database lock-in.
- **Strict YAML Frontmatter:** Machine-readable metadata anchored at the top of each document. Per canonical Google specification, **only a single field is strictly mandatory: `type`**.
- **Progressive Disclosure:** Reserved directory catalog (`index.md`) and change lineage (`log.md`) allow agents to iteratively explore knowledge hierarchies with token-budgeted precision before ingesting deep concept bodies.
- **Interconnected Knowledge Graph:** Standard Markdown hyperlinks (`[Customer Orders](/tables/orders.md)`) create directed semantic edges between tables, metrics, and business policies without needing a separate graph database engine.

---

### Notebook Architecture & Workflow

```mermaid
flowchart TD
    subgraph S1["1. File-Based Knowledge Bundle"]
        IDX["index.md (Progressive Root)"]
        LOG["log.md (Audit Changelog)"]
        DOCS["Concepts: /tables, /metrics, /policies"]
    end

    subgraph S2["2. Ingestion & Validation Engine"]
        PARSE["OKFParser: Frontmatter & Hash Extraction"]
        VAL["OKFValidator: Link Integrity & Staleness"]
        GRAPH["NetworkX DiGraph Construction"]
    end

    subgraph S3["3. Agent Exploration & RAG"]
        PROG["Progressive Index Discovery"]
        EXPAND["Graph Neighborhood Expansion (1..N hops)"]
        ASSEM["Provenance-Augmented Prompt Context"]
        RAG["Deterministic / Groq LLM Generation"]
    end

    S1 --> S2
    S2 --> S3
```

Let's begin by configuring our environment and validating all necessary dependencies.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime, timezone
import matplotlib.pyplot as plt

# Ensure repository root is on sys.path
repo_root = Path.cwd()
if (repo_root / "common").exists():
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
elif (repo_root.parent.parent / "common").exists():
    repo_root = repo_root.parent.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

from common.okf import (
    OKFParser,
    OKFConcept,
    OKFProvenance,
    OKFBundle,
    OKFValidator,
    OKFRAGRetriever,
    OKFContextAssembler,
    render_okf_architecture_diagram,
    render_okf_graph_diagram
)
from common.config import settings

print(f"Environment initialized successfully.")
print(f"Repository Root: {repo_root}")
print(f"Active LLM Provider: {settings.llm_provider}")


## 1. Visualizing the OKF Architecture

The diagram below illustrates the four core pillars of the Google Open Knowledge Format:
1. **Canonical Bundle Directory Tree:** File-based organization using plain UTF-8 Markdown paired with YAML frontmatter.
2. **Attestation & Provenance Engine:** Cryptographic SHA-256 hashes, verified authors, and expiration dates.
3. **Progressive Disclosure Graph Navigator:** Cross-reference links enabling agents to traverse from catalog index to specific concepts.
4. **Provenance-Grounded RAG Assembly:** Deterministic prompt assembly injecting strict source attribution tags (`[OKF:/...]`).

In [ ]:
# Generate and display the canonical architectural diagram
diagram_dir = repo_root / "assets" / "diagrams"
diagram_dir.mkdir(parents=True, exist_ok=True)
arch_img_path = diagram_dir / "track11_okf_architecture.png"

render_okf_architecture_diagram(arch_img_path)

# Display the diagram inline
img = plt.imread(str(arch_img_path))
plt.figure(figsize=(12, 7))
plt.imshow(img)
plt.axis("off")
plt.title("Google Open Knowledge Format (OKF) Architectural Flow", fontsize=13, fontweight="bold", pad=10)
plt.show()


## 2. Anatomy of an OKF Document & Parser Mechanics

According to the canonical Google OKF specification, an OKF concept document consists of two parts:
1. **YAML Frontmatter (delimiters `---`):**
   - **`type` (MANDATORY):** Identifies the concept category (`table`, `metric`, `policy`, `service`, `dataset`).
   - `title`: Human-readable display label.
   - `description`: Crisp semantic definition used for high-level agent indexing.
   - `resource`: URI or identifier of the underlying data system (e.g. BigQuery table, Postgres connection, API endpoint).
   - `tags`: Domain labels for search filtering.
   - `owner`: Engineering team or author responsible.
   - `status`: Lifecycle phase (`active`, `deprecated`, `draft`).
   - `stale_after`: Expiration date after which knowledge must be re-audited.
   - `generated`: Attestation dictionary containing generator ID, timestamp, verification officer, and content hash.
2. **Markdown Body:**
   - Free-form human-readable technical documentation, schema definitions, mathematical equations, or SQL snippets.
   - Cross-concept links using standard Markdown links (`[Customer Orders](/tables/orders.md)`).

Let's test the parser on a sample concept and verify that missing the mandatory `type` field produces a strict specification violation.

In [ ]:
sample_valid_doc = """---
type: metric
title: "Customer Lifetime Value"
description: "Predicted net profit attributed to the entire future relationship with a customer."
resource: "bigquery://prod-warehouse/analytics/clv_gold"
tags: ["crm", "finance", "kpi"]
owner: "growth-analytics"
status: "active"
stale_after: "2027-01-01T00:00:00Z"
generated:
  by: "clv-pipeline-agent/v3"
  at: "2026-06-01T12:00:00Z"
  verified_by: "chief-economist"
  confidence: 0.98
---

# Customer Lifetime Value (CLV)

CLV quantifies long-term customer franchise value.

### Calculation
$$\\text{CLV} = \\text{Average Purchase Value} \\times \\text{Purchase Frequency} \\times \\text{Customer Lifespan}$$

### Connected Entities
- Direct transactions: [Customer Orders](/tables/orders.md)
- Customer dimensions: [Customer Profiles](/tables/customers.md)
"""

# Parse document using OKFParser
concept = OKFParser.parse_text(sample_valid_doc, bundle_relative_path="/metrics/clv.md")

print(f"Path: {concept.path}")
print(f"Type: {concept.concept_type} (Valid OKF Type)")
print(f"Title: {concept.title}")
print(f"Resource URI: {concept.resource}")
print(f"Attested By: {concept.provenance.by} (Verified: {concept.provenance.verified_by})")
print(f"SHA-256 Digest: {concept.content_sha256[:16]}...")
print(f"Extracted Outbound Links: {concept.outbound_links}")
print(f"Stale Status: {'STALE' if concept.is_stale() else 'FRESH & VALID'}")


### Enforcing Canonical Specification Constraints

The Google OKF specification strictly mandates `type`. If a producer emits a document lacking `type`, downstream AI agents cannot safely route or categorize the concept.

Let's verify that our parser rejects untyped files with a clear `ValueError`.

In [ ]:
invalid_doc = """---
title: "Untyped Concept"
description: "Missing mandatory type property."
---

# Body
This document fails canonical specification compliance.
"""

try:
    OKFParser.parse_text(invalid_doc, bundle_relative_path="/docs/invalid.md")
    print("ERROR: Untyped document unexpectedly accepted!")
except ValueError as e:
    print(f"Specification Compliance Check PASSED:")
    print(f"Caught expected error: {e}")


## 3. Loading an Enterprise OKF Bundle & Graph Construction

An OKF **Bundle** is an entire directory tree of cross-referenced concept documents.
The bundle contains two reserved files:
- **`index.md`:** The entry point for progressive disclosure. It enumerates concept categories so an AI agent can read high-level summaries without blowing its token budget.
- **`log.md`:** The append-only audit changelog tracking modifications and version history.

Let's inspect the sample enterprise fixture located in `fixtures/sample_okf_bundle/`.

In [ ]:
bundle_dir = repo_root / "fixtures" / "sample_okf_bundle"
bundle = OKFBundle.from_directory(bundle_dir)

print(f"Loaded OKF Bundle from: {bundle_dir.relative_to(repo_root)}")
print(f"Total Concepts Loaded: {len(bundle.concepts)}")
print(f"Concepts List:")
for path, c in sorted(bundle.concepts.items()):
    print(f"  • {path:<30} [{c.concept_type:<8}] - {c.title}")


## 4. Concept Dependency Graph & Topological Metrics

Because OKF concepts reference each other using standard Markdown links (e.g. `[Orders](/tables/orders.md)`), the bundle forms a **directed knowledge graph**:
- **Nodes:** Canonical concepts (`table`, `metric`, `policy`).
- **Directed Edges:** Explicit cross-references and dependency lineages.

Let's calculate the graph's structural metrics (density, PageRank, DAG status) and visualize the concept network.

In [ ]:
metrics = bundle.get_graph_metrics()

print("OKF Concept Graph Structural Metrics:")
print(f"  • Total Nodes: {metrics['nodes']}")
print(f"  • Total Directed Edges: {metrics['edges']}")
print(f"  • Graph Density: {metrics['density']:.4f}")
print(f"  • Directed Acyclic Graph (DAG): {metrics['is_dag']}")
print("\nTop PageRank Concepts (Centrality):")
for path, rank in sorted(metrics['pagerank'].items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  • {path:<30} PageRank = {rank:.4f}")

# Render and plot the concept dependency network
graph_img_path = diagram_dir / "track11_okf_knowledge_graph.png"
render_okf_graph_diagram(bundle, graph_img_path)

g_img = plt.imread(str(graph_img_path))
plt.figure(figsize=(10, 7))
plt.imshow(g_img)
plt.axis("off")
plt.title("OKF Inter-Concept Cross-Reference Graph", fontsize=13, fontweight="bold", pad=10)
plt.show()


## 5. Progressive Disclosure for AI Agents

When an autonomous agent receives a query like *"Can we refund an order from a high-risk platinum customer?"*, loading every table and policy in the enterprise into prompt context wastes tokens and invites distraction.

**Progressive Disclosure** solves this:
1. The agent first inspects `bundle.get_progressive_index()` (a lightweight summary table containing paths, types, and short descriptions).
2. The agent identifies the relevant concept (e.g. `/policies/refund_sla.md`).
3. The agent traverses the 1-hop neighborhood to pull only the strictly related dependencies (`/tables/orders.md`, `/tables/customers.md`, `/metrics/gross_margin.md`).

Let's demonstrate this progressive retrieval workflow.

In [ ]:
# 1. Compact Progressive Index (Token Budget Friendly)
progressive_index = bundle.get_progressive_index()
print(f"--- PROGRESSIVE DISCLOSURE INDEX ({len(progressive_index)} concepts) ---")
for item in progressive_index:
    print(f"[{item['type'].upper():<8}] {item['path']:<26} :: {item['description']}")

# 2. Targeted 1-Hop Neighborhood Traversal from Seed Concept
seed_path = "/policies/refund_sla.md"
neighborhood = bundle.traverse_neighborhood(seed_path, max_hops=1)

print(f"\n--- 1-HOP NEIGHBORHOOD TRAVERSAL FROM '{seed_path}' ---")
print(f"Discovered {len(neighborhood)} connected concepts:")
for c in neighborhood:
    print(f"  -> [{c.concept_type.upper():<7}] {c.path:<26} (Resource: {c.resource or 'N/A'})")


## 6. Bundle Validation & Linting Engine

In production RAG workflows, bad data in the knowledge base guarantees hallucinations.
`OKFValidator` provides deterministic, automated bundle auditing:
- **Missing Required Fields:** Flagging files lacking `type`.
- **Broken Links:** Detecting Markdown links pointing to deleted or non-existent files.
- **Staleness Expiration:** Identifying concepts whose `stale_after` timestamp has lapsed.
- **Tamper Detection:** Verifying that the SHA-256 digest in `generated.content_hash` matches the actual markdown body.
- **Orphan Detection:** Identifying concepts not connected to the root index or any other document.

Let's run a full validation audit on our bundle.

In [ ]:
report = OKFValidator.validate_bundle(bundle)

print(f"--- OKF BUNDLE VALIDATION REPORT ---")
print(f"Bundle Valid: {'PASSED' if report.is_valid else 'FAILED'}")
print(f"Total Files Audited: {report.total_files}")
print(f"Errors: {report.error_count} | Warnings: {report.warning_count}")
print(f"Concept Type Distribution:")
for ctype, count in sorted(report.concepts_by_type.items()):
    print(f"  • {ctype:<10}: {count}")

if report.issues:
    print("\nIssues Logged:")
    for issue in report.issues:
        print(f"  [{issue.severity}] {issue.path}: ({issue.code}) {issue.message}")
else:
    print("\nClean Bundle: Zero specification violations or broken links found!")


## 7. Provenance-Augmented Context Assembly & Grounded RAG

Now let's connect OKF to a complete RAG workflow.
1. **Hybrid Retrieval (`OKFRAGRetriever`):** Combines continuous semantic embeddings with deterministic graph expansion.
2. **Provenance-Augmented Context (`OKFContextAssembler`):** Generates structured context blocks containing author identities, review officers, and freshness statuses.
3. **Auditable Generation:** The LLM cites explicit OKF document paths (`[OKF:/tables/orders.md]`).

Let's test retrieving knowledge for: *"What is the gross margin formula and how do refund policies impact it?"*

In [ ]:
retriever = OKFRAGRetriever(bundle)
query = "What is the gross margin formula and how do refund policies impact it?"

# Retrieve top seed concepts + 1-hop graph expansion
retrieved_concepts = retriever.retrieve(query, top_k=2, expand_graph_hops=1)

print(f"Query: '{query}'")
print(f"Retrieved {len(retrieved_concepts)} grounded concepts:")
for c in retrieved_concepts:
    print(f"  • {c.path} ({c.concept_type}) - {c.title}")

# Assemble structured prompt context
prompt_context = OKFContextAssembler.assemble_context(retrieved_concepts)
print(f"\n--- STRUCTURED PROMPT CONTEXT (First 500 chars) ---\n")
print(prompt_context[:500] + "\n...")


### Generating Grounded Citations (Offline Deterministic Generator)

Let's run our deterministic offline generator to answer the query, strictly grounding the response in the retrieved OKF concepts and outputting verifiable citations.

In [ ]:
result = OKFContextAssembler.generate_grounded_answer(query, retrieved_concepts)

print(f"--- GROUNDED GENERATION RESULT ---")
print(f"Confidence: {result['confidence']:.2f}")
print(f"Sources Cited: {result['citations']}")
print(f"\nResponse Text:\n{result['answer']}")


## 8. Optional Live Groq API Execution

If a valid `GROQ_API_KEY` is configured in your root `.env` file, the cell below will invoke the Groq LLM to answer the query using the exact OKF context block assembled above.

If no key is configured, it will gracefully fall back without breaking execution.

In [ ]:
if settings.groq_api_key:
    try:
        from groq import Groq
        client = Groq(api_key=settings.groq_api_key)
        
        system_prompt = (
            "You are an enterprise AI data assistant. Answer the user's question using ONLY the provided "
            "Open Knowledge Format (OKF) context. You must cite your sources using the explicit bracketed format "
            "[OKF:/path/to/concept.md]. If the information is not in the context, state that you cannot answer."
        )
        user_prompt = f"Context:\n{prompt_context}\n\nQuestion: {query}"
        
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            max_tokens=400,
        )
        print("--- LIVE GROQ LLM RESPONSE ---")
        print(response.choices[0].message.content)
    except Exception as e:
        print(f"Groq API call skipped or encountered error: {e}")
else:
    print("GROQ_API_KEY not configured in .env. Skipping optional cloud inference.")


## 9. Summary & Architectural Lessons Learned

In this track, we explored Google's **Open Knowledge Format (OKF)** and demonstrated its foundational value for agentic RAG:

1. **Format, Not Platform:** OKF requires no specific cloud provider, proprietary vector database, or complex graph engine. It is pure Markdown files with YAML frontmatter.
2. **The LLM-Wiki Pattern:** By organizing organizational knowledge into linked Markdown concepts, humans and AI agents share a common, version-controlled ground truth.
3. **Progressive Disclosure:** Using `index.md` catalogs, agents can explore deep enterprise architectures within strict context window token limits.
4. **Deterministic Lineage:** Provenance metadata (`generated`, `verified_by`, SHA-256 content hashes) ensures end-to-end auditability and guards against subtle hallucinations.
5. **Bridge to Decision Intelligence:** In Tracks 12 and 13, we will integrate OKF with TypeSafe Jev decision engines and LangGraph state machines to drive auditable, high-stakes enterprise decisions.

---
**Next Track:** [Track 12 — TypeSafe Jev Decision Engine](../12-typesafe-jev/README.md)